# Pipeline walkthrough: what every function outputs and why it is used

This notebook breaks the analysis into its parts. Each function the pipeline uses is called directly on **one example run**, its output is printed (keys, shapes, first rows) and plotted, and the text says **why** the pipeline needs it and **where** its output goes. Use it to read the results in the analysis notebooks and the GUIs, and as a map of the code before you change it.

**Code layout** (dependencies point one way only):

```
po2_transients.py  (pt)  data: .mat -> tables, plus every computation on the tables (no plotting)
       ^
po2_plots.py       (pp)  figures: draws pt tables onto matplotlib axes (notebooks and GUIs)
       ^
po2_gui.py               widgets: RunGUI / AllRunsGUI state and callbacks; calls pt for data, pp for drawing
```

**One run through the pipeline** (stages are numbered the same way in `po2_transients.py`):

```
.mat --1 load_synced_run--> sig, po2 --2 preprocess_event_source--> ext --3 detect_events--> events
     --4 extract_event_segments--> windows --5 POINT_METRICS / PO2_EVENT_METRICS--> TransientData (5 tables)
load_run = stages 1-2 (cached by the GUI)      analyze_run = stages 3-5      process_run = both
```

| § | Stage | Main functions |
|---|---|---|
| 0 | Configuration & registries | `EventParams`, `EVENT_METHODS`, `EVENT_METRICS`, `POINT_METRICS`, `PO2_EVENT_METRICS`, `BIN_METRICS` |
| 1 | Load | `load_synced_run` |
| 2 | Preprocess | `filter_pupil`, `pupil_derivative`, `filter_locom`, `filter_sleep_state`, `preprocess_event_source`, `load_run` |
| 3 | Detect | `event_threshold`, `find_event_peaks`, `positive_lobe_bounds`, `EVENT_METRICS`, `detect_events` |
| 4 | Extract | `extract_event_segments` |
| 5 | pO2 metrics | `po2_point_qc`, `po2_variants`, `normalize_event_segments`, `po2_event_metrics` |
| 6 | Run tables | `analyze_run`, `TransientData` |
| 7 | All runs + QC | `process_all_runs`, `point_qc` |
| 8 | Event averages | `interpolate_event`, `event_responses`, `group_average` |
| 9 | Binning (AllRunsGUI) | `classify_events`, `bin_labels`, `bar_values`, `correlation_values` |
| 10 | Mixed model | `event_point_table`, `fit_mixed_model`, `extract_fixed_effects`, `extract_variance_decomposition` |
| 11 | Reference | GUI panel → function map, how to add a metric, function index |

**How to use this notebook.** Read it top to bottom and run each cell as you go. It is the "open the hood" companion to the analysis notebooks: nothing here is needed to *get* results, but it shows what each function returns. Every function shown here lives in [po2_transients.py](../po2_transients.py) (`pt`) or [po2_plots.py](../po2_plots.py) (`pp`). In VS Code, Cmd-click (Ctrl-click on Windows) a function name in a code cell to jump to its source. Hover over it to see its documentation.

## Before you run this

> **New to Python or Jupyter?** Do the one-time setup in the [README](../README.md#full-setup-from-zero) first: install conda, create the `po2` environment, put the data in `data/`.

1. **Choose the kernel.** Top-right of this notebook in VS Code, click *Select Kernel* → *Python Environments* → **po2**. (Jupyter Lab: *Kernel → Change Kernel → Python (po2)*.) The kernel is the Python process that runs the cells. Picking `po2` makes it use the packages from `environment.yml`.
2. **Run cells top to bottom.** Click in a cell and press **Shift + Enter** to run it and move to the next one, or use **Run All**. Cells share variables like the MATLAB workspace, so a cell only works once the cells above it have run.
3. **Changed a parameter?** Re-run that cell and every cell below it. If things get confusing, **Restart** the kernel (it clears all variables, like `clear all`) and run everything again.
4. Figures and tables appear under the cell that made them. Text printed in red/pink is usually a *warning* (it keeps going), not an *error* (it stops, with a "Traceback" ending in the error message).
5. The configuration cell below picks the example run (`EXAMPLE`) and the event method. Set `event_method="pupil"` or `"locom"` to follow the other event types through the same steps.

## 0. Configuration and registries

`params` holds every tunable number. `EXAMPLE` picks the run used in §1–§6. Set `params.event_method = "locom"` to walk through locomotion events instead (§2 shows both methods regardless).

The two helpers below only format output for this notebook: `describe` lists the entries of a dict (type, shape, first values) and `peek` shows array-valued table cells as their shape.

In [ ]:
# %matplotlib inline: draw figures as static images under each cell (the GUI notebook uses "widget" instead)
%matplotlib inline
import inspect
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

import po2_transients as pt   # this repo's DATA layer (tables + computations)
import po2_plots as pp        # this repo's FIGURE layer

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)

runs = {
    "26-10-05_B6_318": [5, 6, 8, 9, 11, 12, 14],
    "26-10-07_B6_322": [6, 12],
    "26-10-21_B6_322": [2, 3],
}
params = pt.EventParams(
    event_method="sleep_state", rel_to_rms=False, min_distance_s=10, extract_window_s=(-30, 30),height_thresh=.1, neg_peaks=True, # all events
    movmedian_len_s=6, lpf_cutoff_hz=0.1, # pupil
    locom_sigma_s=2.0, # locomotion
)
print("Reading session folders from:", pt.DATA_DIR)
EXAMPLE = ("26-10-05_B6_318", 6)       # (session folder, run) walked through in sections 1-6
method = pt.EVENT_METHODS[params.event_method]


def describe(d):
    # one row per dict entry: type, shape, dtype and the first values (scalars shown as is)
    rows = []
    for k, v in d.items():
        if np.isscalar(v):
            rows.append({"key": k, "type": type(v).__name__, "shape": "", "preview": f"{v:.4g}" if isinstance(v, float) else v})
        else:
            a = np.asarray(v)
            rows.append({"key": k, "type": f"{type(v).__name__} {a.dtype}", "shape": a.shape,
                         "preview": np.array2string(a.ravel()[:4], precision=4, separator=", ")})
    return pd.DataFrame(rows).set_index("key")


def peek(df, n=5):
    # first n rows, with every array-valued cell replaced by "array(shape)"
    out = df.head(n).copy()
    for c in out.columns:
        if len(out) and isinstance(out[c].iloc[0], np.ndarray):
            out[c] = [f"array{np.shape(v)}" for v in out[c]]
    return out


print(params.describe())
print(method)

**Registries** are the extension points of the pipeline. Each is a dict, and adding an entry is enough: the tables, `BIN_METRICS`, the plots and the GUI dropdowns all read from them.

- `EVENT_METHODS`: which signal events are detected on, and how it is filtered.
- `EVENT_METRICS`: one scalar per event, as `events` columns.
- `POINT_METRICS`: one scalar per run × pO2 point, as `po2_signals` columns.
- `PO2_VARIANTS`: whole-run pO2 traces, as `po2_signals` columns.
- `PO2_EVENT_METRICS`: one scalar per event × point, as `po2_events` columns.

`BIN_METRICS` is built from the three metric registries. It lists every metric that events can be binned by.

In [ ]:
def doc(fn):
    return (inspect.getdoc(fn) or "").split("\n")[0]

registry_rows = (
    [{"registry": "EVENT_METHODS", "name": k, "description": f"raw={m.raw}, signals={m.signals}, detect_on={m.detect_on!r}, "
                                                            f"preprocess={m.preprocess.__name__}"} for k, m in pt.EVENT_METHODS.items()]
    + [{"registry": "EVENT_METRICS", "name": k, "description": doc(f)} for k, f in pt.EVENT_METRICS.items()]
    + [{"registry": "POINT_METRICS", "name": k, "description": doc(f)} for k, f in pt.POINT_METRICS.items()]
    + [{"registry": "PO2_VARIANTS", "name": k, "description": doc(f)} for k, f in pt.PO2_VARIANTS.items()]
    + [{"registry": "PO2_EVENT_METRICS", "name": k, "description": doc(f)} for k, f in pt.PO2_EVENT_METRICS.items()]
)
with pd.option_context("display.max_colwidth", 120):
    display(pd.DataFrame(registry_rows))
print("BIN_METRICS (metric -> table):", pt.BIN_METRICS)
print("TABLE_KEYS  (table -> row keys):", pt.TABLE_KEYS)

## 1. Load: `load_synced_run(folder, run, raw_names)`

**What:** reads `pO2_all_synced_runXX.mat`. Returns `sig`, a dict with the pO2 time base and the requested raw event-source signals (`RAW_SIGNALS` maps each name to its .mat variables), and `po2`, a `(n_time, n_points)` matrix with one column per measurement point along the vessel.

**Why:** this is the only place the pipeline touches the file. Everything after it is numpy.

**Where it goes:** `sig` becomes the single row of the `dataset` table and feeds stage 2. `po2` feeds stage 5 (`po2_signals`) and stage 4 (pO2 event windows).

Note the very different sampling rates: the behavior signals are fast and pO2 is slow (< 1 Hz). Later steps convert seconds to samples separately for each signal.

In [ ]:
folder, run = EXAMPLE
sig, po2 = pt.load_synced_run(folder, run, raw_names=["pupil", "locom", "sleep_state"])   # both sources, to show both methods
display(describe(sig))
print(f"po2: shape {po2.shape} (n_time, n_points), NaN fraction {np.isnan(po2).mean():.3f}, "
      f"duration {sig['t_po2'][-1] - sig['t_po2'][0]:.0f} s")

fig, axs = plt.subplots(4, 1, figsize=(11, 7), sharex=True, layout="constrained")
axs[0].plot(sig["t_pupil"], sig["pupil"], color=pp.COLORS["raw"], lw=0.8)
axs[0].set_ylabel("raw pupil")

axs[1].plot(sig["t_locom"], sig["locom"], color=pp.COLORS["raw"], lw=0.3)
axs[1].set_ylabel("raw locom (accel)")

axs[2].plot(sig["t_sleep_state"], sig["sleep_state"], color=pp.COLORS["raw"], lw=0.8)
axs[2].set_ylabel("raw sleep state")

im = axs[3].imshow(po2.T, aspect="auto", origin="lower", interpolation="nearest",
                   extent=[sig["t_po2"][0], sig["t_po2"][-1], -0.5, po2.shape[1] - 0.5])
fig.colorbar(im, ax=axs[3], label="pO2")
axs[3].set_ylabel("pO2 point")
axs[3].set_xlabel("Time (s)")

fig.suptitle(f"Stage 1 output: {folder} run {run}")
plt.show()

## 2. Preprocess: from raw signal to detection signal

Each `EVENT_METHODS` entry names a `preprocess(sig, params)` function. It returns `ext`, a dict with `t_<s>, <s>` for every filtered signal `s` of the method. One of those signals, `detect_on`, is the one events are detected on.

### 2a. Pupil (`filter_pupil`, `pupil_derivative`)
**What:** a moving median (`movmedian_len_s`) removes blinks and tracking spikes. A zero-phase 4th-order Butterworth low-pass (`lpf_cutoff_hz`) then smooths the trace. Its first difference `dxdt_pupil` peaks where the pupil dilates fastest.

**Why:** dilation *onsets* are the events. Without the median filter, a blink would be the largest "dilation". `filtfilt` runs the filter forward and backward, so there is no phase lag that would shift event times.

The median-only trace below is recomputed here to show the intermediate step. `filter_pupil` does both steps in one call.

In [ ]:
pupil_med = median_filter(sig["pupil"], size=int(params.movmedian_len_s * sig["fs_pupil"]))    # 1st half of filter_pupil
pupil_filt = pt.filter_pupil(sig["pupil"], sig["fs_pupil"], params.movmedian_len_s, params.lpf_cutoff_hz)
t_dxdt, dxdt = pt.pupil_derivative(sig["t_pupil"], pupil_filt)

print(f"filter_pupil -> {pupil_filt.shape};  pupil_derivative -> t {t_dxdt.shape}, dxdt {dxdt.shape} (one sample shorter)")

fig, axs = plt.subplots(2, 1, figsize=(11, 5), sharex=True, layout="constrained")
axs[0].plot(sig["t_pupil"], sig["pupil"], color=pp.COLORS["raw"], lw=0.8, label="raw")
axs[0].plot(sig["t_pupil"], pupil_med, color="orange", lw=1, label=f"moving median ({params.movmedian_len_s} s)")
axs[0].plot(sig["t_pupil"], pupil_filt, color=pp.COLORS["filt"], lw=2, label=f"+ low-pass ({params.lpf_cutoff_hz} Hz) = filter_pupil")
axs[0].legend(fontsize=8, loc="upper right"); axs[0].set_ylabel("pupil (a.u.)")

axs[1].plot(t_dxdt, dxdt, color=pp.COLORS["dxdt"]); axs[1].axhline(0, color="k", lw=0.5)
axs[1].set_ylabel("dPupil/dt (per sample)"); axs[1].set_xlabel("Time (s)")

plt.show()

### 2b. Locomotion (`filter_locom`)
**What:** subtracts the mean (a crude high-pass), rectifies and Gaussian-smooths with `locom_sigma_s`. The result is a movement envelope, and events are its peaks.

**Why:** the raw accelerometer swings positive and negative at 400 Hz. Its envelope is a slow, positive "how much is the mouse moving" signal that can be thresholded like the pupil derivative.

In [ ]:
locom_filt = pt.filter_locom(sig["locom"], sig["fs_locom"], params.locom_sigma_s)
fig, ax = plt.subplots(figsize=(11, 3), layout="constrained")
ax.plot(sig["t_locom"], np.abs(sig["locom"] - sig["locom"].mean()), color=pp.COLORS["raw"], lw=0.3, label="|accel - mean|")
ax.plot(sig["t_locom"], locom_filt, color=pp.COLORS["dxdt"], lw=2, label=f"Gaussian sigma {params.locom_sigma_s} s = filter_locom")
ax.legend(fontsize=8, loc="upper right"); ax.set_xlabel("Time (s)"); ax.set_ylabel("locomotion")
plt.show()

### 2c. Sleep state (`filter_sleep_state`, `sleep_state_derivative`)
**What:** cleans the scored sleep-state trace (removes bouts shorter than `min_distance_s`, fills gaps as described below). Its first difference is nonzero only at state transitions, and those are the events (`neg_peaks=True` keeps transitions in both directions).

    A "bout" is a stretch of one nonzero state; 0s inside it do not break it but do not count towards its
    duration (duration = number of nonzero samples / fs).
      - Bouts shorter than min_distance_s -> 0 (unknown).
      - 0s embedded in a kept bout -> that bout's state.
      - 0s between two kept bouts -> the state of the later bout.
      - 0s next to only one kept bout (the other side is a dropped bout or the recording edge) stay 0.
      - Transition REM from 3 -> 4 (important later)
    Single pass: bouts are not re-merged after short bouts between them are dropped.

In [ ]:
sleep_state_filt = pt.filter_sleep_state(sig["sleep_state"], sig["fs_sleep_state"], params.min_distance_s)
t_dxdt, dxdt = pt.sleep_state_derivative(sig["t_sleep_state"], sleep_state_filt)

fig, axs = plt.subplots(2, 1, figsize=(11, 6), layout="constrained")
axs[0].plot(sig["t_sleep_state"], sig["sleep_state"], color=pp.COLORS["raw"], lw=0.8, label="raw")
axs[0].plot(sig["t_sleep_state"], sleep_state_filt, color=pp.COLORS["filt"], lw=2, label=f"filtered (min {params.min_distance_s} s)")
axs[0].legend(fontsize=8); axs[0].set_ylabel("sleep state"); axs[0].set_xlabel("Time (s)")

axs[1].plot(t_dxdt, dxdt, color=pp.COLORS["dxdt"], lw=2, label="dxdt_sleep_state")
axs[1].legend(fontsize=8, loc="upper right"); axs[1].set_ylabel("d sleep state / dt (per sample)"); axs[1].set_xlabel("Time (s)")

plt.show()

### 2d. `preprocess_event_source(sig, params)` and `load_run(folder, run, params)`
**What:** `preprocess_event_source` calls `EVENT_METHODS[params.event_method].preprocess` and returns `(ext, fs)`, where `fs` is the detection signal's sampling rate. `load_run` packages stages 1 and 2 into a `LoadedRun` (keys, sig, po2, ext, fs).

**Why:** loading and filtering are the slow part. The `RunGUI` caches a `LoadedRun` per run, so a threshold change only re-runs `analyze_run` (stages 3–5).

**Where it goes:** `ext` becomes the `extract_signals` table and is the input of detection and extraction.

In [ ]:
ext, fs = pt.preprocess_event_source(sig, params)
print(f"event method {params.event_method!r}: detection signal {method.detect_on!r} at fs = {fs:g} Hz")
display(describe(ext))

loaded = pt.load_run(folder, run, params)
print("LoadedRun fields:", [f for f in vars(loaded)], "| keys:", loaded.keys)

## 3. Detect events

### 3a. `event_threshold(signal, height_thresh)`
**What:** `height_thresh × RMS(signal)`. **Why:** the signal amplitude differs from run to run (pupil size, camera distance), so a threshold relative to each run's RMS keeps the detection comparable. With `rel_to_rms=False`, `height_thresh` is used as an absolute value instead.

In [ ]:
if params.rel_to_rms:
    t_det, x_det = loaded.ext["t_" + method.detect_on], loaded.ext[method.detect_on]
    rms = np.sqrt(np.nanmean(x_det ** 2))
    threshold = pt.event_threshold(x_det, params.height_thresh)
    print(f"RMS({method.detect_on}) = {rms:.4g}  ->  threshold = {params.height_thresh} x RMS = {threshold:.4g}")
else:
    t_det, x_det = loaded.ext["t_" + method.detect_on], loaded.ext[method.detect_on]
    threshold = pt.event_threshold(x_det, params.height_thresh)
    print(f"threshold = {threshold:.4g}")

### 3b. `find_event_peaks(...)` → `DetectionContext` and `positive_lobe_bounds`
**What:** `scipy.signal.find_peaks` with `height=threshold` and `distance=min_distance_s` (peaks closer than this keep only the largest). `positive_lobe_bounds` then finds, for each peak, the stretch where the signal stays > 0 around it (the last sample ≤ 0 before the peak and the first one after). Each side of that lobe is capped at `min_distance_s` from the peak.

**Why:** the lobe is the whole dilation (or movement bout) the peak belongs to, which `event_auc` integrates over. The `DetectionContext` bundles everything a metric could need: the signal, `fs`, the peak indices, the lobe bounds and the `find_peaks` properties.

In [ ]:
ctx, thr = pt.find_event_peaks(t_det, x_det, fs, params.height_thresh, params.min_distance_s, params.rel_to_rms, neg_peaks=params.neg_peaks)
uncapped = pt.positive_lobe_bounds(x_det, ctx.peaks)
print("DetectionContext.props keys:", list(ctx.props))
pd.DataFrame({
    "peak idx": ctx.peaks, "peak time (s)": t_det[ctx.peaks],
    "lobe (uncapped, s)": [f"{t_det[s]:.1f} - {t_det[e]:.1f}" for s, e in uncapped],
    "lobe (capped, s)": [f"{t_det[s]:.1f} - {t_det[e]:.1f}" for s, e in ctx.bounds],
    "peak height": ctx.props["peak_heights"], "width (samples)": ctx.props["widths"],
})

### 3c. `EVENT_METRICS`: one scalar per event
**What:** each registered function takes the `DetectionContext` and returns one value per event:
- `event_height`: the detection signal at the peak.
- `event_halfwidth`: the peak width at half prominence, in seconds.
- `event_auc`: the trapezoid area of the signal over the lobe. For `dxdt_pupil` this is the total pupil change over the dilation.

**Where it goes:** `events` columns, `BIN_METRICS` (so they can be binned in `AllRunsGUI` / `classify_events`) and the `AllRunsGUI` bars and correlation.

The zoom below draws each metric on one event: the dot is the height, the horizontal bar the half-width and the shaded area the AUC.

In [ ]:
metrics = {name: fn(ctx) for name, fn in pt.EVENT_METRICS.items()}
display(pd.DataFrame(metrics))

if len(ctx.peaks):
    k = np.random.randint(len(ctx.peaks))          # random event
    p, (s, e) = ctx.peaks[k], ctx.bounds[k]
    pad = int(1.5 * params.min_distance_s * fs)
    sl = slice(max(p - pad, 0), min(p + pad, len(x_det)))
    fig, ax = plt.subplots(figsize=(11, 3.5), layout="constrained")
    ax.plot(t_det[sl], x_det[sl], color=pp.COLORS["dxdt"], label=method.detect_on)
    ax.axhline(thr if not params.neg_peaks else -thr, color=pp.COLORS["threshold"], ls="--", label="threshold")
    ax.axhline(0, color="k", lw=0.5)
    ax.fill_between(t_det[s:e + 1], 0, x_det[s:e + 1], color=pp.COLORS["lobe"], alpha=0.4,
                    label=f"lobe -> event_auc = {metrics['event_auc'][k]:.3g}")
    ax.scatter(t_det[p], x_det[p], color="k", zorder=5, label=f"event_height = {metrics['event_height'][k]:.3g}")
    li, ri = ctx.props["left_ips"][k], ctx.props["right_ips"][k]       # interpolated half-prominence crossings
    ax.hlines(ctx.props["width_heights"][k] if x_det[p] > 0 else -ctx.props["width_heights"][k],
              np.interp(li, np.arange(len(t_det)), t_det),
              np.interp(ri, np.arange(len(t_det)), t_det), color="k", lw=2,
              label=f"event_halfwidth = {metrics['event_halfwidth'][k]:.3g} s")
    ax.legend(fontsize=8); ax.set_xlabel("Time (s)"); ax.set_title(f"Event {k}: what each EVENT_METRICS entry measures")
    plt.show()

### 3d. `detect_events(...)`: stage 3 in one call
**What:** `find_event_peaks` plus every `EVENT_METRICS` entry, plus `event_time` and `event_bounds` converted to seconds. **This is what `analyze_run` calls.** The standard figure of this stage is `pp.plot_run_events`, shown in §6 (and `pp.plot_detection_panel` in `RunGUI` [0, 0]).

In [ ]:
ev, threshold = pt.detect_events(t_det, x_det, fs, params.height_thresh, params.min_distance_s, rel_to_rms=params.rel_to_rms, neg_peaks=params.neg_peaks)
print(f"{len(ev['event_time'])} events, threshold {threshold:.4g}")
describe(ev)

## 4. Extract: `extract_event_segments(t, data, t_events, extract_window_s)`

**What:** for each event, it takes the samples `searchsorted(t_event) + arange(w0·fs, w1·fs)` and returns `t_rel` `(n_window,)` and `segments` `(n_events, n_window)`. Samples outside the recording are NaN, so every window has the same length.

**Why:** event-locked windows are what is averaged and compared. The same call runs on every method signal (→ `events.<s>_event_trace`) and on every pO2 point (→ `po2_events.event_trace`). Because each signal keeps its own `fs`, the windows have different lengths (compare the sample counts below). §8 puts them on a common grid.

**Known issue A** (left as is so results stay reproducible): sample `searchsorted(t_event)` is the first one *at or after* the event, but it is labeled `t_rel = 0`. The table shows that lag, which can be close to one pO2 sample.

In [ ]:
t_events = ev["event_time"]
t_rel_b, seg_b = pt.extract_event_segments(t_det, x_det, t_events, params.extract_window_s)
point = 0
t_rel_p, seg_p = pt.extract_event_segments(loaded.sig["t_po2"], loaded.po2[:, point], t_events, params.extract_window_s)
print(f"{method.detect_on}: t_rel {t_rel_b.shape}, segments {seg_b.shape}, NaN samples {np.isnan(seg_b).sum()}")
print(f"pO2 point {point}: t_rel {t_rel_p.shape}, segments {seg_p.shape}, NaN samples {np.isnan(seg_p).sum()}")
lag = loaded.sig["t_po2"][loaded.sig["t_po2"].searchsorted(t_events)] - t_events
display(pd.DataFrame({"event_time": t_events, "pO2 lag of t_rel=0 sample (s)": lag, "pO2 sample period (s)": 1 / loaded.sig["fs_po2"]}))

fig, axs = plt.subplots(1, 2, figsize=(12, 3.5), layout="constrained")
for seg in seg_b:
    axs[0].plot(t_rel_b, seg, alpha=0.6)
axs[0].set_title(f"{len(seg_b)} {method.detect_on} windows ({seg_b.shape[1]} samples each)")
for seg in seg_p:
    axs[1].plot(t_rel_p, seg, "o-", ms=3, alpha=0.6)
axs[1].set_title(f"pO2 point {point} windows ({seg_p.shape[1]} samples each)")
for ax in axs:
    ax.axvline(0, color="k", ls=":"); ax.set_xlabel(pp.event_xlabel(params))
plt.show()

## 5. pO2 metrics

### 5a. `po2_point_qc(po2)` → `POINT_METRICS` (one scalar per point, whole run)
**What:**
- `rms`: sqrt(mean(pO2²)), the magnitude.
- `cv`: std / mean, the noise level.
- `baseline`: the mean.
- `nan_frac`: the fraction of NaN samples.

**Why:** `cv` drives the point QC cutoff (`max_cv` in the analysis notebooks). `rms` normalizes each point's event windows. `baseline` normalizes `norm_po2`.

**Where it goes:** `po2_signals` columns, `pt.point_qc(data)`, and the `RunGUI` [1, 1] bars drawn by `pp.plot_point_bars`, below.

In [ ]:
qc = pt.po2_point_qc(loaded.po2)
display(pd.DataFrame(qc).T.round(3))
run_tables = pt.analyze_run(loaded, params)          # (stage 3-5 tables; explained in section 6) for the pp plots
fig = plt.figure(figsize=(11, 4), layout="constrained")
ax_rms, ax_cv = fig.subplots(2, 1, sharex=True)
pp.plot_point_bars(ax_rms, ax_cv, run_tables.po2_signals, selected=range(loaded.po2.shape[1]),
                   colors=pp.point_colors(loaded.po2.shape[1]))
plt.show()

### 5b. `po2_variants(po2, qc)` → `PO2_VARIANTS` (whole-run traces)
**What:** `po2` (raw), `abs_po2` and `norm_po2 = po2 / baseline`. **Where it goes:** `po2_signals` columns. `RunGUI` [1, 0] shows `po2` or `norm_po2`.

Two normalizations exist and they are different. `norm_po2` divides by the **mean**: 1 = the point's average level. The event windows of §5c use `(trace − rms) / rms`, so 0 = the point's **RMS**, which sits slightly above the mean (RMS/mean = sqrt(1 + cv²)).

In [ ]:
variants = pt.po2_variants(loaded.po2, qc)
display(describe(variants))
print(f"RMS / mean per point: min {np.nanmin(qc['rms'] / qc['baseline']):.3f}, "
      f"max {np.nanmax(qc['rms'] / qc['baseline']):.3f}  (= sqrt(1 + cv^2))")

### 5c. `normalize_event_segments` and `po2_event_metrics` → `PO2_EVENT_METRICS` (per event × point)
**What:** each point's windows are normalized to `(window − rms) / rms`, so points with different absolute pO2 share one scale (→ `po2_events.event_trace_norm`). From each normalized window:
- `event_peak`: the signed value at the largest |deviation| over the **whole** window, before or after the event.
- `peak_abs_height`: its absolute value.
- `event_peak_time`: when it happens relative to the event.

**Where it goes:** `po2_events` columns, `BIN_METRICS`, the `AllRunsGUI` bars and correlation, and the outcome of the mixed model (§10).

In [ ]:
norm = pt.normalize_event_segments(seg_p, qc["rms"][point])
pm = pt.po2_event_metrics(t_rel_p, norm)
display(pd.DataFrame(pm))

fig, ax = plt.subplots(figsize=(11, 3.5), layout="constrained")
for i, w in enumerate(norm):
    line, = ax.plot(t_rel_p, w, "o-", ms=3, alpha=0.7, label=f"event {i}")
    ax.scatter(pm["event_peak_time"][i], pm["event_peak"][i], s=90, facecolor="none", edgecolor=line.get_color(), lw=2)
ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", ls=":")
ax.set_ylabel(pp.SIGNAL_LABELS["po2_norm"]); ax.set_xlabel(pp.event_xlabel(params))
ax.set_title(f"Point {point}: normalized windows, circles = event_peak at event_peak_time"); ax.legend(fontsize=7, ncol=6)
plt.show()

## 6. Run tables: `analyze_run(loaded, params)` → `TransientData`

**What:** stages 3–5 on a `LoadedRun`, assembled into the five tables. Every table starts with the run keys, and `pt.TABLE_KEYS` gives the columns that identify a row, so tables join on them. `process_run(folder, run, params)` is `analyze_run(load_run(...))`.

**Where it goes:** everything downstream: the analysis notebooks, `pp` plots and both GUIs.

In [ ]:
display(run_tables.summary())
for name in ["dataset", "po2_signals", "extract_signals", "events", "po2_events"]:
    print(f"\n{name}  (row keys: {pt.TABLE_KEYS[name]})")
    display(peek(getattr(run_tables, name), 3))

Joining on the keys: each `po2_events` row (event × point) gets its event's `event_auc` and its point's `cv`. This is how cross-table questions such as "do bigger dilations give bigger pO2 peaks at clean points?" are asked. `pt.correlation_values` and `pt.bin_labels` (§9) do this merging for the GUI.

The standard figure of stages 2–3 for one run is `pp.plot_run_events`. In it, the shaded windows are the extraction windows, the salmon spans are the lobes, and the green dashed line is the threshold.

In [ ]:
joined = (run_tables.po2_events[pt.EVENT_POINT_KEYS + ["event_peak"]]
          .merge(run_tables.events[pt.EVENT_KEYS + ["event_auc"]], on=pt.EVENT_KEYS)
          .merge(run_tables.po2_signals[pt.POINT_KEYS + ["cv"]], on=pt.POINT_KEYS))
display(joined.head())
pp.plot_run_events(run_tables, folder, run, params)
plt.show()

## 7. All runs: `process_all_runs(runs, params)` and point QC

**What:** `process_run` for every run, with the tables stacked. `point_qc(data)` gives the run × point quality columns. Points with `cv > max_cv` are noisy and can be excluded by passing `points=qc_points` to the averaging, plotting and model functions.

In [ ]:
data = pt.process_all_runs(runs, params)
display(data.summary())
max_cv = 1
points = pt.point_qc(data)
qc_points = points[points["cv"] <= max_cv]
print(f"{len(qc_points)}/{len(points)} pO2 points pass CV <= {max_cv}")
pp.plot_qc_histograms(points, data.events, max_cv, event_label=pp.signal_label(method.detect_on))
plt.show()

## 8. Event averages: `interpolate_event` → `event_responses` → `group_average`

Runs have different sampling rates (pO2 at 0.92 or 0.33 Hz), so their windows do not share sample times and cannot be averaged sample by sample. Every average in the pipeline therefore works in three steps:

1. `interpolate_event(t_rel, trace, window, dt)` resamples one window onto `arange(w0, w1 + dt/2, dt)`. It never extrapolates: grid times outside the finite samples are NaN. 2-D pO2 traces are averaged over points first.
2. `event_responses(data, signal, window, dt, level)` does this for every window and returns a **long table**: keys + `[t, value]`. With `level="event"` there is one trace per event (pO2 averaged over the event's points, so each event counts once). With `level="point"` there is one trace per event × point.
3. `group_average(responses, labels, by)` gives mean, std and n per group and `t`. `labels` attaches a grouping column (e.g. an event class) on the shared keys.

**Where it goes:** `pp.plot_pooled_events`, `plot_class_averages`, `plot_point_class_averages`, `plot_group_average` and the `AllRunsGUI` [1, 0] bin averages.

In [ ]:
row = data.po2_events.iloc[0]
t_grid, values = pt.interpolate_event(row.t_event, row.event_trace_norm, params.extract_window_s, dt=1.0)
fig, ax = plt.subplots(figsize=(11, 3), layout="constrained")
ax.plot(row.t_event, row.event_trace_norm, "o-", color=pp.COLORS["raw"], label=f"observed ({len(row.t_event)} samples at run fs)")
ax.plot(t_grid, values, ".-", color=pp.COLORS["filt"], label=f"interpolate_event ({len(t_grid)} grid points, dt = 1 s)")
ax.legend(fontsize=8); ax.set_xlabel(pp.event_xlabel(params)); ax.set_ylabel(pp.SIGNAL_LABELS["po2_norm"])
plt.show()

resp_event = pt.event_responses(data, "po2_norm", params.extract_window_s, dt=1.0, level="event", points=qc_points)
resp_point = pt.event_responses(data, "po2_norm", params.extract_window_s, dt=1.0, level="point", points=qc_points)
print(f"event_responses level='event': {len(resp_event)} rows = {resp_event.groupby(pt.EVENT_KEYS).ngroups} events x {resp_event.t.nunique()} t")
print(f"event_responses level='point': {len(resp_point)} rows = {resp_point.groupby(pt.EVENT_POINT_KEYS).ngroups} event x point x {resp_point.t.nunique()} t")
display(resp_event.head())
display(pt.group_average(resp_event).head())
display(pt.group_average(resp_event, by="mouse_id").head())

In [ ]:
pp.plot_pooled_events(data, params, dt=1.0, points=qc_points)
plt.show()
pp.plot_group_average(pt.group_average(resp_event, by="mouse_id"), "mouse_id", "po2_norm", xlabel=pp.event_xlabel(params))
plt.show()

## 9. Binning: what `AllRunsGUI` computes

### 9a. `bin_edges`, `bin_index`, `classify_events`
**What:** `bins` are `[lo, hi]` pairs (inclusive), given as percentiles (`percentile=True`) or as metric values. `bin_edges` converts them to metric units. `bin_index` returns each value's bin, or −1; where bins overlap, the later one wins. `classify_events` returns an ordered categorical with labels like `event_auc P0-33`.

**Known issue B:** `DEFAULT_BINS = [[0, 33], [34, 67], [68, 100]]` leaves gaps at P33–34 and P67–68. Rows that fall in a gap are in no bin (a warning says how many).

`values_to_percentiles` is the inverse of `bin_edges`. The GUI uses it to convert typed bounds when *Percentiles* is toggled.

In [ ]:
metric, bins = "event_height", pt.DEFAULT_BINS
values = pt.metric_table(data, metric)[metric]
edges = pt.bin_edges(values, bins, percentile=True)
print("percentile bins", bins, "-> edges in", metric, "units:\n", edges.round(4))
print("round trip values_to_percentiles(edges):\n", pt.values_to_percentiles(edges, values).round(2))
classes = pt.classify_events(data.events, metric, bins)
classes.value_counts(sort=False)

### 9b. `bin_labels(data, metric, bins)` → which rows each bin holds
**What:** returns `(labels, per_point)`.
- An `events` metric gives one bin per **event**, keyed by `EVENT_KEYS`.
- A `po2_events` metric gives one bin per **event × point** row.
- A `po2_signals` metric (e.g. `rms`) gives every event × point row its **point's** bin.

`per_point` says which level the bins live at, and so whether pO2 averages use `level="point"`.

In [ ]:
for m in ["event_auc", "rms", "event_peak"]:
    lab, per_point = pt.bin_labels(data, m, bins)
    print(f"{m:11s} ({pt.BIN_METRICS[m]:11s}) -> labels {lab.shape}, per_point={per_point}, "
          f"rows per bin {lab['bin'].value_counts(sort=False).to_dict()}, unbinned {lab['bin'].isna().sum()}")
labels, per_point = pt.bin_labels(data, metric, bins)
labels.head()

### 9c. `bar_values`, `group_average(…, labels, "bin")`, `correlation_values`: the four AllRunsGUI panels
- **[0, 0]** `pp.plot_bin_histogram`: the bin metric's distribution colored by bin.
- **[0, 1]** `bar_values(data, bar_metric, labels, per_point)`: the bar metric of every binned row with its bin → `pp.plot_bin_bars`.
- **[1, 0]** `group_average(event_responses(...), labels, "bin")` → `pp.plot_bin_traces`. Behavior traces always use `level="event"`. pO2 traces use `level="point"` when `per_point`.
- **[1, 1]** `correlation_values(data, x, y, bin_metric, bins)`: x and y merged on their shared keys, with `color_idx` = bin, `NO_BIN` (−1) or `MULTI_BIN` (−2, a dot that matches rows of several bins) → `pp.plot_correlation`.

The figure below is the AllRunsGUI with its default settings, drawn statically.

In [ ]:
bar = pt.bar_values(data, "event_peak", labels, per_point)
corr = pt.correlation_values(data, "rms", "event_peak", metric, bins)
behavior_avg = pt.group_average(pt.event_responses(data, method.detect_on, params.extract_window_s, 1.0), labels, "bin")
po2_avg = pt.group_average(pt.event_responses(data, "po2", params.extract_window_s, 1.0, level="point" if per_point else "event"),
                           labels, "bin")
print(f"bar_values: {bar.shape}; correlation_values: {corr.shape}, color_idx counts {corr.color_idx.value_counts().to_dict()}")
display(bar.head(3)); display(corr.head(3))

colors = pp.bin_colors(len(bins))
fig = plt.figure(figsize=(13, 8), layout="constrained")
axs = fig.subplots(2, 2)
pp.plot_bin_histogram(axs[0, 0], values, edges, colors, metric, n_runs=len(data.dataset))
pp.plot_bin_bars(axs[0, 1], bar, colors, "event_peak", metric, unit="event x point rows")
grid = axs[1, 0].get_subplotspec().subgridspec(2, 1); axs[1, 0].remove()
ax_b, ax_p = fig.add_subplot(grid[0]), fig.add_subplot(grid[1])
pp.plot_bin_traces(ax_b, behavior_avg, colors, pp.signal_label(method.detect_on))
pp.plot_bin_traces(ax_p, po2_avg, colors, pp.SIGNAL_LABELS["po2"]); ax_p.set_xlabel(pp.event_xlabel(params))
pp.plot_correlation(axs[1, 1], corr, colors, "rms", "event_peak", metric)
fig.suptitle(pp.sample_title(data, params), fontsize=9)
plt.show()

## 10. Mixed model: does the pO2 peak depend on the event class?

- `event_point_table(data, points)` gives one row per event × QC point with the `PO2_EVENT_METRICS`, plus `run_id`, `event_id` and `point_id`. These are unique across sessions, because run, event and point numbers repeat.
- `fit_mixed_model` fits `outcome ~ C(event_class)` with a random intercept per mouse and nested variance components.
- `extract_fixed_effects` gives the class contrasts with Wald CIs (→ `pp.plot_fixed_effects`).
- `extract_variance_decomposition` gives the share of variance per component (→ `pp.plot_variance_decomposition`).

> ⚠️ **Known: with `event_method="sleep_state"` (the setting in §0) this cell stops with `LinAlgError: Singular matrix`.** Sleep-state `event_auc` takes only two values, so the lowest percentile class is empty and the model can't be fitted. Set `event_method="pupil"` in the §0 configuration cell and re-run to see this section work. The same thing happens in `05_analyze_sleep_po2_transients.ipynb`.

In [ ]:
data.events["event_class"] = pt.classify_events(data.events, "event_auc", pt.DEFAULT_BINS)
lmm_df = pt.event_point_table(data, points=qc_points).merge(data.events[pt.EVENT_KEYS + ["event_class"]], on=pt.EVENT_KEYS)
display(lmm_df.head())
fit, model_df, formula = pt.fit_mixed_model(lmm_df, "event_peak", ["event_class"], ["date", "run_id", "event_id", "point_id"], "mouse_id")
print("formula:", formula)
effects = pt.extract_fixed_effects(fit)
variance = pt.extract_variance_decomposition(fit, model_df, formula, group_name="mouse_id")
display(effects); display(variance)
pp.plot_fixed_effects(effects, title="event_peak: event class effects", xlabel="Difference vs. lowest class")
pp.plot_variance_decomposition(variance, title="event_peak: variance decomposition")
plt.show()

## 11. Reference

### GUI panels → functions

| GUI panel | Data (`pt`) | Drawing (`pp`) |
|---|---|---|
| RunGUI [0, 0] detection | `load_run` (cached) → `analyze_run` | `plot_detection_panel` |
| RunGUI [1, 0] pO2 traces | `po2_signals[po2 / norm_po2]` | `plot_po2_traces` |
| RunGUI [1, 1] point bars | `po2_signals[rms, cv]` | `plot_point_bars` |
| RunGUI [0, 1] pO2 event windows | `po2_event_traces(run, variant)`, `trace_mean_std` | `plot_po2_events` |
| AllRunsGUI [0, 0] histogram | `metric_table`, `bin_edges`, `bin_index` | `plot_bin_histogram` |
| AllRunsGUI [0, 1] bin bars | `bin_labels` → `bar_values` | `plot_bin_bars` |
| AllRunsGUI [1, 0] bin averages | `event_responses` → `group_average(labels, "bin")` | `plot_bin_traces` |
| AllRunsGUI [1, 1] correlation | `correlation_values` | `plot_correlation` |

### Adding a metric (example)
A new per-event metric only needs a function of the `DetectionContext` and one registry line. Here the peak prominence (which `find_peaks` already computes) is registered for this session, and `analyze_run` gains the column. If the line goes in `po2_transients.py` instead, the metric also appears in `BIN_METRICS` and so in every GUI dropdown, because `BIN_METRICS` is built at import. The other registries work the same way (see the "How to extend" section of the module docstring).

In [ ]:
def event_prominence(ctx):
    # prominence of each peak, from scipy.signal.find_peaks
    return ctx.props["prominences"]

pt.EVENT_METRICS["event_prominence"] = event_prominence
demo = pt.analyze_run(loaded, params)
display(peek(demo.events[pt.EVENT_KEYS + list(pt.EVENT_METRICS)]))
del pt.EVENT_METRICS["event_prominence"]       # undo, so the rest of the session is unchanged

### Function index
The first docstring line of every public function, grouped by module. The docstrings in the source give the arguments, return shapes and which table or plot uses each one.

In [ ]:
def index(module):
    rows = [{"function": name, "summary": doc(fn)} for name, fn in inspect.getmembers(module, inspect.isfunction)
            if fn.__module__ == module.__name__ and not name.startswith("_")]
    return pd.DataFrame(rows)

with pd.option_context("display.max_colwidth", 140, "display.max_rows", 200):
    print("po2_transients (pt)"); display(index(pt))
    print("po2_plots (pp)"); display(index(pp))